[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/estimation/pose_diffusion/station_keeping.ipynb)

# Station Keeping in Gusts

This example computes how far a vessel on dynamic positioning wanders from its set point. The vessel holds a fixed spot with its thrusters while wind and waves push it around, so it never sits exactly on its set point: it wanders, and more sideways than forward, because gusts hit the side of the hull harder than the bow. Its spread determines how much clear water the vessel needs around it and how stiff the thruster controller has to be; it is predicted directly, checked against a simulated fleet, and used to choose a controller gain.

The vessel's pose error is a twist, the small motion from the set point to where the vessel is, and a linear measurement of that error is a line, paired with it by `line & error`. Their spread is a covariance map taking each measurement line to the twist correlated with it, so `line & covariance(line)` is the variance of the measurement. The controller and any steady turn define a linear drift map on twists, which acts on the covariance from both sides—on the twists it returns and, through `dynamics.adjugate()`, on the lines it takes—and the steady spread where the controller removes uncertainty as fast as the gusts add it is found in one linear solve, batched over every controller gain at once.

In matrix notation the error reads as a coordinate vector $(v_x, v_y, \omega)$ in a chosen order, the covariance reads as a symmetric $3\times3$ matrix $P$ with $\dot P = F P + P F^{\mathsf T} + Q$, and the steady spread is the solution of the continuous Lyapunov equation $F P + P F^{\mathsf T} + Q = 0$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext, stack
from numga.algebras import PGA2D as ga
from examples.util.animation import save_figure
from examples.estimation.pose_diffusion import render

np.set_printoptions(precision=3, suppress=True)

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                         # Scalar+
# A small motion combining rotation and translation is a twist bivector, and a linear measurement of a
# twist is an antibivector line.
Twist = ga.gatype.bivector()                        # Bivector+
Line = ga.gatype.antibivector()                     # Antibivector-
Covariance = ga.gatype((Twist, Line))
Dynamics = ga.gatype((Twist, Twist))
Spread = ga.gatype((Scalar, Line, Line))
Gusts = ga.gatype((Twist, Line))
# The commanded set point at the origin.
ORIGIN = mv.w.dual()                                # [] Point-

## 1. Gusts and the controller

The vessel's error is a twist taking the set point to where the vessel actually is. A gust is modelled as white noise sent through a linear map from lines to twists, built as a sum of dyads over the directions gusts push in—sideways, forward, and in yaw—each at its own strength. The controller is a linear map on twists, `Twist.commutator(still) - gain * Twist`: it reduces the error by a fixed fraction per second, so the error decays with a time constant of one over the gain, while the commutator turns the error along with the vessel when the vessel turns on the spot.

In matrix notation one chooses an order for the twist's coordinates, say $(v_x, v_y, \omega)$, and writes the dynamics as $F = \operatorname{ad}_\Omega - k\,I$, with $\operatorname{ad}_\Omega$ the $3\times3$ matrix of the commutator with the turning rate in that order, signs included, and the gust map as a $3\times3$ matrix $B$.

In [ ]:
# The directions gusts push in, in the vessel's frame (sideways, forward, and in yaw), and their
# strengths per square root of a second in metres, metres, and radians; and the controller gain (1/s).
pushes = stack([mv.yw, mv.wx, mv.xy])                                         # [pushes] Twist+
strengths = np.array([0.5, 0.15, 0.01])
gain = 0.1

In [ ]:
# Each push direction at its strength, read off the white noise by the line it pairs with.
gusts = (pushes * strengths * (pushes & Line)).sum(axis=-1)                   # [] Twist+ <- Line-
# The vessel keeps its heading, and the controller shrinks the error at gain per second.
still = mv.xy * 0.0                                                           # [] Twist+
controller = Twist.commutator(still) - gain * Twist                           # [] Twist+ <- Twist+

## 2. The spread as a covariance

The spread of the vessel's pose error is a covariance map from lines to twists. A line measures the error as the scalar `line & error`, and the covariance returns the twist correlated with that measurement so that `line & covariance(line)` is the variance of the measurement. White noise is defined in an orthonormal frame of lines with unit variance along each and no correlation between them, so the covariance the gusts add per second is the sum over that frame of each line's gust dyad.

In [ ]:
def covariance(gusts: Gusts) -> Covariance:
    """The noise covariance added per second by white-noise gusts."""
    # White noise has unit variance along each axis of an orthonormal line frame, so each axis adds its gust dyad.
    frame = mv.antibivector(np.eye(3))                                        # [axes] Line-
    return (gusts(frame) * (gusts(frame) & Line)).sum(axis=0)                 # [] Twist+ <- Line-


gust_covariance = covariance(gusts)                                           # [] Twist+ <- Line-
print("gust covariance per second, a map from lines to twists:\n", gust_covariance.coefficient_table())

## 3. The steady state in one solve

As the errors change, so does their covariance. The dynamics act on the twists the covariance returns, and they also change which errors a given line picks out: measuring the changed error `dynamics(t)` with a line gives the same number as measuring `t` itself with `dynamics.adjugate()(line)`, the map on lines that incidence carries over from the dynamics. The covariance's rate of change is the sum of these two effects and the gusts, `dynamics(covariance) + covariance(dynamics.adjugate()) + noise`. Once the controller removes spread as fast as the gusts add it, the covariance stops growing. Because the rate of change is linear in the covariance, writing the covariance as a sum of dyads turns the growth into a linear map of both twists and the line, and matching its line input against `-noise` solves for both twist inputs at once.

In matrix notation the covariance is a symmetric $3\times3$ matrix $P$ with rate of change $\dot P = F P + P F^{\mathsf T} + Q$, where $Q = B B^{\mathsf T}$ and the transpose $F^{\mathsf T}$ corresponds to `dynamics.adjugate()`. Setting $\dot P = 0$ gives the continuous Lyapunov equation $F P + P F^{\mathsf T} + Q = 0$, solved either by vectorizing into the $9\times9$ Kronecker system $(I \otimes F + F \otimes I)\,\operatorname{vec} P = -\operatorname{vec} Q$ or by the Bartels–Stewart algorithm.

In [ ]:
def settled(dynamics: Dynamics, noise: Covariance) -> Covariance:
    """The steady-state covariance at which the controller removes uncertainty as fast as the gusts add it."""
    # Writing the covariance as a dyad t * (s & l) turns its rate of change into a linear map of both twists and the line.
    lyapunov = dynamics(Twist) * (Twist & Line) + Twist * (dynamics(Twist) & Line)   # [...] Twist+ <- (Twist+, Twist+, Line-)
    # Match the line input against -noise and solve for both twist inputs; Line.dual() feeds a line in as a twist.
    return lyapunov.lstsq(-noise)(Line.dual())                                       # [...] Twist+ <- Line-


steady = settled(controller, gust_covariance)                                 # [] Twist+ <- Line-

## 4. Checking against a fleet

A simulated fleet checks the prediction. In each step every vessel's error changes under the controller and receives an independent gust, while the predicted covariance is advanced by its rate of change over the same step. Because white noise integrated over a step has variance proportional to the step's length, the gusts are scaled by the square root of `dt`. In the figure, each vessel is drawn as a short arrow along its heading, overlaid with the integrated prediction (solid) and the steady state from the solve (dashed) as two-standard-deviation position ellipses.

In [ ]:
# The number of simulated vessels, the run duration and time step in seconds, the noise seed, and the
# half-width of the plot in metres.
n_vessels = 2000
seconds, dt = 150.0, 0.2
seed = 0
extent = 5.0

In [ ]:
def simulate(
    dynamics: Dynamics, noise: Covariance, n_vessels: int, seconds: float, dt: float, seed: int,
) -> tuple[Twist, Covariance]:
    """The pose errors and integrated covariance of a fleet of vessels starting at the set point."""
    rng = np.random.default_rng(seed)
    # All vessels start at the set point with zero spread.
    errors = mv.bivector(np.zeros((n_vessels, 3)))                            # [vessels] Twist+
    predicted = noise * 0.0                                                   # [] Twist+ <- Line-
    for _ in range(int(seconds / dt)):
        # White noise integrated over the step, and the dynamics acting on the covariance's twist output and line input.
        white = mv.antibivector(rng.normal(size=(n_vessels, 3))) * np.sqrt(dt)  # [vessels] Line-
        errors = errors + dynamics(errors) * dt + gusts(white)                # [vessels] Twist+
        growth = dynamics(predicted) + predicted(dynamics.adjugate()) + noise  # [] Twist+ <- Line-
        predicted = predicted + growth * dt                                   # [] Twist+ <- Line-
    return errors, predicted


errors, predicted = simulate(controller, gust_covariance, n_vessels, seconds, dt, seed)
poses = (errors * 0.5).exp()                                                  # [vessels] Motor+
save_figure(render.draw_settled({"holding heading": (poses, predicted, steady)}, extent), "station_keeping");

## 5. Choosing the controller gain

A stiffer controller keeps the vessel closer to the set point at the cost of more thruster work. Because the steady state is a single linear solve, it can be evaluated for a whole array of gains at once: the gain becomes a batch of scalars, and every map broadcasts over it. The position spread follows from how an error moves the set point, `Twist.commutator(ORIGIN)`, whose adjugate pulls each displacement-measuring line back to a twist-measuring line. Requiring the vessel to keep its two-standard-deviation excursion within 2 m in every direction determines the smallest gain that meets the tolerance; in the plot, the red curve traces the long-axis half-width, the blue curve traces the short-axis half-width, and the dashed lines mark the 2 m tolerance and the minimum gain.

In matrix notation the position covariance reads as $\Sigma = J P J^{\mathsf T}$, with $J$ the $2\times3$ Jacobian of the set point's displacement with respect to the twist coordinates; with the set point at the rotation centre, $J$ selects the two translation coordinates.

In [ ]:
# The controller gains (1/s) to sweep across, and the allowed two-standard-deviation excursion in metres.
n_gains = 60
gains = np.linspace(0.02, 0.5, n_gains)
tolerance = 2.0

In [ ]:
def position_spread(covariance: Covariance) -> Spread:
    """The symmetric covariance form on lines measuring displacements of the set point, pulled back from the twist covariance."""
    # An error moves the set point by their commutator; its adjugate pulls a displacement-measuring line back to a twist measurement.
    shift = Twist.commutator(ORIGIN)                                          # [] Point- <- Twist+
    readout = shift.adjugate()                                                # [] Line- <- Line+
    return readout & covariance(readout)                                      # [...] Scalar+ <- (Line+, Line+)


gain_controllers = Twist.commutator(still) - mv.scalar(gains[:, None]) * Twist  # [gains] Twist+ <- Twist+
spreads = position_spread(settled(gain_controllers, gust_covariance))         # [gains] Scalar+ <- (Line+, Line+)
save_figure(render.draw_envelope(gains, spreads, tolerance), "station_keeping_gain");

Along the vessel's side the two-standard-deviation excursion is the gust strength divided by the square root of twice the gain, so the 2 m tolerance requires a gain of 0.125 per second, an 8 s time constant. Forward, where gusts push less, the same gain leaves a wide margin.

## 6. Turning on the spot

A vessel that turns while holding station carries its error around with it: a sideways push becomes a forward error a quarter turn later, where gusts push less. The commutator with the turning rate adds this rotation to the drift map while leaving the rest of the calculation unchanged. At one revolution per minute the position spread becomes rounder and shrinks along its long axis.

In [ ]:
# One full turn per minute, and the seed of the turning fleet's disturbances.
turning_rate = mv.xy * (2 * np.pi / 60)                                       # [] Twist+
turning_seed = 1

In [ ]:
turning_controller = Twist.commutator(turning_rate) - gain * Twist            # [] Twist+ <- Twist+
turning_errors, turning_predicted = simulate(turning_controller, gust_covariance, n_vessels, seconds, dt, turning_seed)
save_figure(render.draw_settled({"holding heading": (poses, predicted, steady),
                                 "one turn a minute": ((turning_errors * 0.5).exp(), turning_predicted, settled(turning_controller, gust_covariance))}, extent),
            "station_keeping_turning");

In [ ]:
# checks
from examples.estimation.pose_diffusion import core

np.testing.assert_allclose((gust_covariance - core.covariance(gusts)).kernel, 0.0, atol=1e-12)
np.testing.assert_allclose((steady - core.settled(controller, gust_covariance)).kernel, 0.0, atol=1e-12)
np.testing.assert_allclose((spreads - core.position_spread(core.settled(gain_controllers, gust_covariance))).kernel, 0.0, atol=1e-12)
# The steady state has zero rate of change, the integrated prediction reaches it, and the fleet matches it.
steady_growth = controller(steady) + steady(controller.adjugate()) + gust_covariance
np.testing.assert_allclose(steady_growth.kernel, 0.0, atol=1e-10)
np.testing.assert_allclose(predicted.kernel, steady.kernel, atol=1e-6)
empirical = (errors * (errors & Line)).mean(axis=0)                           # [] Twist+ <- Line-
np.testing.assert_allclose(empirical.kernel, steady.kernel, atol=0.1 * np.abs(steady.kernel).max())
# Sideways the variance equals the gust strength squared divided by twice the gain.
np.testing.assert_allclose(position_spread(steady).eigvalsh().to_array()[-1], strengths[0]**2 / (2 * gain), rtol=1e-8)